# Task 1: Fine-Tuning Flan-T5-Small for Scientific Question Answering

**Student:** Avinash Megavatn (ID: 16829749)

**Module:** 7043SCN - Generative AI and Reinforcement Learning

**Model:** google/flan-t5-small (80M params, encoder-decoder)

**Dataset:** allenai/sciq

**Technique:** LoRA (Low-Rank Adaptation)

In [1]:
import subprocess, sys
for pkg in ["torch", "transformers>=4.36.0", "datasets>=2.16.0", "peft>=0.7.0",
            "accelerate>=0.25.0", "evaluate>=0.4.0", "rouge-score", "nltk",
            "sentencepiece", "protobuf", "scikit-learn", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

All packages installed.


In [2]:
import os, time, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import torch
import nltk
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = {
    "model_name": "google/flan-t5-small",
    "dataset_name": "allenai/sciq",
    "max_input_length": 512, "max_target_length": 128,
    "batch_size": 8, "learning_rate": 3e-4, "num_epochs": 3,
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.1,
    "output_dir": "./results", "seed": SEED,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [3]:
dataset = load_dataset(CONFIG["dataset_name"])
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]}")

split_dataset = {
    "train": dataset["train"].shuffle(seed=SEED).select(range(min(4000, len(dataset["train"])))),
    "validation": dataset["validation"].shuffle(seed=SEED).select(range(min(500, len(dataset["validation"])))),
    "test": dataset["test"].shuffle(seed=SEED).select(range(min(500, len(dataset["test"])))),
}
print(f"Using - Train: {len(split_dataset['train'])}, Val: {len(split_dataset['validation'])}, Test: {len(split_dataset['test'])}")

Train: 11679, Val: 1000, Test: 1000
Sample: {'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'distractor3': 'viruses', 'distractor1': 'protozoa', 'distractor2': 'gymnosperms', 'correct_answer': 'mesophilic organisms', 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}
Using - Train: 4000, Val: 500, Test: 500


In [4]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def preprocess(examples):
    inputs = []
    for q, s in zip(examples["question"], examples["support"]):
        ctx = s[:300] if s else ""
        if ctx:
            inputs.append(f"Answer the science question given the context.\nContext: {ctx}\nQuestion: {q}")
        else:
            inputs.append(f"Answer the science question: {q}")
    targets = examples["correct_answer"]
    model_inputs = tokenizer(inputs, max_length=CONFIG["max_input_length"], truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=CONFIG["max_target_length"], truncation=True, padding="max_length")
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = {k: v.map(preprocess, batched=True, remove_columns=v.column_names) for k, v in split_dataset.items()}
print("Tokenization complete.")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.


In [5]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    exact_match = sum(1 for p, r in zip(decoded_preds, decoded_labels) if p.lower().strip() == r.lower().strip()) / len(decoded_preds)
    bleu_scores = []
    for p, r in zip(decoded_preds, decoded_labels):
        try:
            pt, rt = p.split(), r.split()
            bleu_scores.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
        except: bleu_scores.append(0.0)
    return {k: round(v, 4) for k, v in {"rouge1": rouge["rouge1"], "rouge2": rouge["rouge2"],
            "rougeL": rouge["rougeL"], "bleu": np.mean(bleu_scores), "exact_match": exact_match}.items()}

## Baseline Evaluation (Zero-Shot)

In [6]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"]).to(device)
model_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model parameters: {model_params:,} ({model_params/1e6:.1f}M)")

baseline_preds, baseline_refs, test_questions = [], [], []
baseline_model.eval()
t0 = time.time()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        ctx = sample["support"][:300] if sample["support"] else ""
        if ctx:
            inp_text = f"Answer the science question given the context.\nContext: {ctx}\nQuestion: {sample['question']}"
        else:
            inp_text = f"Answer the science question: {sample['question']}"
        inp = tokenizer(inp_text, return_tensors="pt", max_length=CONFIG["max_input_length"], truncation=True).to(device)
        out = baseline_model.generate(**inp, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        baseline_preds.append(tokenizer.decode(out[0], skip_special_tokens=True).strip())
        baseline_refs.append(sample["correct_answer"].strip())
        test_questions.append(sample["question"])
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(split_dataset['test'])}")
baseline_eval_time = time.time() - t0

bl_rouge = rouge_metric.compute(predictions=baseline_preds, references=baseline_refs, use_stemmer=True)
bl_em = sum(1 for p, r in zip(baseline_preds, baseline_refs) if p.lower().strip() == r.lower().strip()) / len(baseline_preds)
bl_bleu = []
for p, r in zip(baseline_preds, baseline_refs):
    try:
        pt, rt = p.split(), r.split()
        bl_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: bl_bleu.append(0.0)

baseline_metrics = {"rouge1": round(bl_rouge["rouge1"],4), "rouge2": round(bl_rouge["rouge2"],4),
                    "rougeL": round(bl_rouge["rougeL"],4), "bleu": round(np.mean(bl_bleu),4), "exact_match": round(bl_em,4)}
print(f"Baseline: R1={baseline_metrics['rouge1']}, R2={baseline_metrics['rouge2']}, RL={baseline_metrics['rougeL']}, BLEU={baseline_metrics['bleu']}, EM={baseline_metrics['exact_match']}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}")
    print(f"Pred: {baseline_preds[i][:100]}")
    print(f"Ref: {baseline_refs[i][:100]}")

del baseline_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model parameters: 76,961,152 (77.0M)


  100/500


  200/500


  300/500


  400/500


  500/500


Baseline: R1=0.6791, R2=0.1921, RL=0.6774, BLEU=0.0, EM=0.564

Q: What functions in removing phosphorylated amino acids from proteins?
Pred: phosphatase
Ref: phosphatase

Q: What disease is unpreventable in the type one form but may be prevented by diet if it is of the second type?
Pred: Type 1 diabetes
Ref: diabetes

Q: Melting ice is drastically impacting the number of what at glacier national park?
Pred: active glaciers
Ref: active glaciers


## Hardware Assessment

Full fine-tuning of Flan-T5-Small (80M parameters) is more feasible than larger models but still requires substantial compute. On CPU-only hardware, training would take 8-10+ hours for 3 epochs with full parameter updates. We use **LoRA (Low-Rank Adaptation)** to reduce trainable parameters by ~99%, enabling efficient training on CPU in under 1 hour while maintaining competitive performance.

## LoRA Fine-Tuning

In [7]:
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])
lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=CONFIG["lora_r"],
                         lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
                         target_modules=["q", "v"], bias="none")
model = get_peft_model(model, lora_config)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
reduction = (1 - trainable/total_params) * 100
print(f"LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, targets=[q, v]")
print(f"Total: {total_params:,}, Trainable: {trainable:,} ({trainable/total_params*100:.4f}%), Reduction: {reduction:.2f}%")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LoRA: r=16, alpha=32, targets=[q, v]
Total: 77,649,280, Trainable: 688,128 (0.8862%), Reduction: 99.11%


In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"], num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"], per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"], weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True,
    predict_with_generate=True, generation_max_length=CONFIG["max_target_length"],
    logging_steps=50, report_to="none", seed=SEED, fp16=torch.cuda.is_available(), dataloader_num_workers=0,
    optim="adamw_torch",
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training: {CONFIG['num_epochs']} epochs, batch={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}")
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
print(f"Done in {train_time:.1f}s ({train_time/60:.1f} min), loss={train_result.training_loss:.4f}")

epoch_metrics = [e for e in trainer.state.log_history if "eval_rouge1" in e]
for em in epoch_metrics:
    print(f"  Epoch {em['epoch']:.0f}: Loss={em['eval_loss']:.4f}, R1={em['eval_rouge1']:.4f}, RL={em['eval_rougeL']:.4f}, EM={em.get('eval_exact_match', 0):.4f}")

Training: 3 epochs, batch=8, lr=0.0003


Epoch,Training Loss,Validation Loss


## LoRA Evaluation on Test Set

In [ ]:
lora_preds = []
model.eval()
t0 = time.time()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        ctx = sample["support"][:300] if sample["support"] else ""
        if ctx:
            inp_text = f"Answer the science question given the context.\nContext: {ctx}\nQuestion: {sample['question']}"
        else:
            inp_text = f"Answer the science question: {sample['question']}"
        inp = tokenizer(inp_text, return_tensors="pt", max_length=CONFIG["max_input_length"], truncation=True).to(device)
        out = model.generate(**inp, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        lora_preds.append(tokenizer.decode(out[0], skip_special_tokens=True).strip())
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(split_dataset['test'])}")
lora_eval_time = time.time() - t0

lora_rouge = rouge_metric.compute(predictions=lora_preds, references=baseline_refs, use_stemmer=True)
lora_em = sum(1 for p, r in zip(lora_preds, baseline_refs) if p.lower().strip() == r.lower().strip()) / len(lora_preds)
lora_bleu = []
for p, r in zip(lora_preds, baseline_refs):
    try:
        pt, rt = p.split(), r.split()
        lora_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: lora_bleu.append(0.0)

lora_metrics = {"rouge1": round(lora_rouge["rouge1"],4), "rouge2": round(lora_rouge["rouge2"],4),
                "rougeL": round(lora_rouge["rougeL"],4), "bleu": round(np.mean(lora_bleu),4), "exact_match": round(lora_em,4)}
print(f"LoRA: R1={lora_metrics['rouge1']}, R2={lora_metrics['rouge2']}, RL={lora_metrics['rougeL']}, BLEU={lora_metrics['bleu']}, EM={lora_metrics['exact_match']}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}")
    print(f"LoRA: {lora_preds[i][:100]}")
    print(f"Ref: {baseline_refs[i][:100]}")

## Comparative Analysis and Visualization

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "Exact Match"],
    "Baseline": [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"], baseline_metrics["exact_match"]],
    "LoRA": [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"], lora_metrics["exact_match"]],
})
comparison["Improvement"] = comparison["LoRA"] - comparison["Baseline"]
comparison["Improvement_%"] = ((comparison["Improvement"] / comparison["Baseline"].replace(0,1)) * 100).round(2)
print(comparison.to_string(index=False))
print(f"\nTotal params: {total_params:,}, Trainable: {trainable:,}, Reduction: {reduction:.2f}%")
print(f"Training: {train_time:.1f}s, Baseline eval: {baseline_eval_time:.1f}s, LoRA eval: {lora_eval_time:.1f}s")

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "Exact Match"]
bl_vals = [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"], baseline_metrics["exact_match"]]
lo_vals = [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"], lora_metrics["exact_match"]]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.arange(len(metrics_names)); w = 0.35
axes[0,0].bar(x-w/2, bl_vals, w, label="Baseline", color="#3498db", edgecolor="black", linewidth=0.5)
axes[0,0].bar(x+w/2, lo_vals, w, label="LoRA", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(metrics_names, fontsize=9)
axes[0,0].set_title("Baseline vs LoRA"); axes[0,0].legend(); axes[0,0].grid(axis="y", alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    axes[0,1].plot(eps, [e["eval_loss"] for e in epoch_metrics], "b-o", lw=2)
    axes[0,1].set_title("Validation Loss"); axes[0,1].set_xlabel("Epoch"); axes[0,1].grid(alpha=0.3)
    axes[1,0].plot(eps, [e["eval_rouge1"] for e in epoch_metrics], "r-o", lw=2, label="R1")
    axes[1,0].plot(eps, [e["eval_rouge2"] for e in epoch_metrics], "g-s", lw=2, label="R2")
    axes[1,0].plot(eps, [e["eval_rougeL"] for e in epoch_metrics], "b-^", lw=2, label="RL")
    axes[1,0].set_title("ROUGE Scores"); axes[1,0].set_xlabel("Epoch"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].bar(["Total", "Trainable (LoRA)"], [total_params, trainable], color=["#3498db", "#e74c3c"], edgecolor="black")
axes[1,1].set_title("Parameter Efficiency"); axes[1,1].set_yscale("log"); axes[1,1].grid(axis="y", alpha=0.3)

plt.suptitle("Flan-T5-Small Science Q&A: LoRA Fine-Tuning Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/results.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# ============================================================
# Figure 1: Improvement Analysis (Absolute + Percentage)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_vals = [lo - bl for lo, bl in zip(lo_vals, bl_vals)]
colors_imp = ["#2ecc71" if v >= 0 else "#e74c3c" for v in imp_vals]
axes[0].bar(metrics_names, imp_vals, color=colors_imp, edgecolor="black", linewidth=0.5)
axes[0].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(imp_vals):
    axes[0].text(i, v + (0.005 if v >= 0 else -0.02), f"{v:+.4f}", ha="center", fontsize=8, fontweight="bold")
axes[0].set_title("Absolute Improvement (LoRA - Baseline)", fontsize=11)
axes[0].set_ylabel("Score Difference"); axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", rotation=20)

pct_vals = [(lo - bl) / bl * 100 if bl != 0 else 0 for lo, bl in zip(lo_vals, bl_vals)]
colors_pct = ["#2ecc71" if v >= 0 else "#e74c3c" for v in pct_vals]
axes[1].bar(metrics_names, pct_vals, color=colors_pct, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(pct_vals):
    axes[1].text(i, v + (2 if v >= 0 else -5), f"{v:+.1f}%", ha="center", fontsize=8, fontweight="bold")
axes[1].set_title("Percentage Improvement", fontsize=11)
axes[1].set_ylabel("Improvement (%)"); axes[1].grid(axis="y", alpha=0.3)
axes[1].tick_params(axis="x", rotation=20)

plt.suptitle("Flan-T5-Small Science Q&A: Improvement Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/improvement_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 2: Training Dynamics (Step-Level Loss + Epoch Metrics)
# ============================================================
train_steps = [e for e in trainer.state.log_history if "loss" in e and "eval_loss" not in e]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_steps:
    steps = [e["step"] for e in train_steps]
    losses = [e["loss"] for e in train_steps]
    axes[0].plot(steps, losses, color="#e74c3c", linewidth=1.0, alpha=0.5, label="Raw Loss")
    if len(losses) > 5:
        w = min(10, max(3, len(losses)//3))
        ma = np.convolve(losses, np.ones(w)/w, mode="valid")
        axes[0].plot(steps[w-1:], ma, color="#2c3e50", linewidth=2.5, label=f"Moving Avg (w={w})")
    axes[0].set_title("Training Loss per Step"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    ax2 = axes[1].twinx()
    axes[1].plot(eps, [e.get("eval_rouge1", 0) for e in epoch_metrics], "r-o", lw=2, markersize=7, label="ROUGE-1")
    axes[1].plot(eps, [e.get("eval_rougeL", 0) for e in epoch_metrics], "b-^", lw=2, markersize=7, label="ROUGE-L")
    axes[1].plot(eps, [e.get("eval_exact_match", 0) for e in epoch_metrics], "g-s", lw=2, markersize=7, label="Exact Match")
    ax2.plot(eps, [e["eval_loss"] for e in epoch_metrics], "k--o", lw=1.5, markersize=5, alpha=0.5, label="Eval Loss")
    ax2.set_ylabel("Eval Loss")
    axes[1].set_ylabel("Score"); axes[1].set_xlabel("Epoch")
    axes[1].set_title("Validation Metrics Across Epochs")
    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, loc="center right", fontsize=8)
    axes[1].grid(alpha=0.3)

plt.suptitle("Flan-T5-Small Science Q&A: Training Dynamics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_dynamics.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 3: Prediction Analysis (Length Distribution + BLEU Box)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bl_lens = [len(p.split()) for p in baseline_preds]
lo_lens = [len(p.split()) for p in lora_preds]
ref_lens = [len(r.split()) for r in baseline_refs]

axes[0].hist(ref_lens, bins=20, alpha=0.4, label=f"Reference (mean={np.mean(ref_lens):.1f})", color="#2ecc71", edgecolor="black")
axes[0].hist(bl_lens, bins=20, alpha=0.5, label=f"Baseline (mean={np.mean(bl_lens):.1f})", color="#3498db", edgecolor="black")
axes[0].hist(lo_lens, bins=20, alpha=0.5, label=f"LoRA (mean={np.mean(lo_lens):.1f})", color="#e74c3c", edgecolor="black")
axes[0].set_title("Prediction Length Distribution"); axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

bp = axes[1].boxplot([bl_bleu, lora_bleu], labels=["Baseline", "LoRA"], patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
bp["boxes"][0].set_facecolor("#3498db"); bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor("#e74c3c"); bp["boxes"][1].set_alpha(0.6)
axes[1].set_title("Per-Sample BLEU Distribution"); axes[1].set_ylabel("BLEU Score"); axes[1].grid(alpha=0.3)

plt.suptitle("Flan-T5-Small Science Q&A: Prediction Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/prediction_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 4: Summary Dashboard (Radar + Pie + Timing + Heatmap)
# ============================================================
fig = plt.figure(figsize=(16, 12))

ax1 = fig.add_subplot(2, 2, 1, polar=True)
N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
ax1.plot(angles, bl_vals + bl_vals[:1], "o-", lw=2, label="Baseline", color="#3498db")
ax1.fill(angles, bl_vals + bl_vals[:1], alpha=0.15, color="#3498db")
ax1.plot(angles, lo_vals + lo_vals[:1], "o-", lw=2, label="LoRA", color="#e74c3c")
ax1.fill(angles, lo_vals + lo_vals[:1], alpha=0.15, color="#e74c3c")
ax1.set_xticks(angles[:-1]); ax1.set_xticklabels(metrics_names, fontsize=8)
ax1.set_title("Performance Radar", fontsize=12, pad=20)
ax1.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)

ax2 = fig.add_subplot(2, 2, 2)
frozen = total_params - trainable
ax2.pie([trainable, frozen], labels=[f"Trainable\n({trainable:,})", f"Frozen\n({frozen:,})"],
        colors=["#e74c3c", "#3498db"], autopct="%1.2f%%", startangle=90,
        textprops={"fontsize": 9}, explode=[0.05, 0])
ax2.set_title("LoRA Parameter Distribution", fontsize=12)

ax3 = fig.add_subplot(2, 2, 3)
times = [baseline_eval_time, train_time, lora_eval_time]
time_labels = ["Baseline\nEval", "LoRA\nTraining", "LoRA\nEval"]
bars = ax3.bar(time_labels, times, color=["#3498db", "#e74c3c", "#2ecc71"], edgecolor="black", linewidth=0.5)
for bar, t in zip(bars, times):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(times)*0.02,
             f"{t:.1f}s", ha="center", fontsize=10, fontweight="bold")
ax3.set_title("Timing Comparison"); ax3.set_ylabel("Time (seconds)"); ax3.grid(axis="y", alpha=0.3)

ax4 = fig.add_subplot(2, 2, 4)
heatmap_data = np.array([bl_vals, lo_vals])
sns.heatmap(heatmap_data, annot=True, fmt=".4f", xticklabels=metrics_names,
            yticklabels=["Baseline", "LoRA"], cmap="YlOrRd", ax=ax4,
            linewidths=0.5, linecolor="black", cbar_kws={"shrink": 0.8})
ax4.set_title("Metrics Heatmap", fontsize=12)

plt.suptitle("Flan-T5-Small Science Q&A: Summary Dashboard", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 5: BLEU Score Deep Analysis (CDF + Scatter)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sorted_bl = np.sort(bl_bleu)
sorted_lo = np.sort(lora_bleu)
axes[0].plot(sorted_bl, np.linspace(0, 1, len(sorted_bl)), lw=2, label="Baseline", color="#3498db")
axes[0].plot(sorted_lo, np.linspace(0, 1, len(sorted_lo)), lw=2, label="LoRA", color="#e74c3c")
axes[0].set_title("Cumulative BLEU Distribution (CDF)"); axes[0].set_xlabel("BLEU Score")
axes[0].set_ylabel("Cumulative Proportion"); axes[0].legend(); axes[0].grid(alpha=0.3)

min_len = min(len(bl_bleu), len(lora_bleu))
axes[1].scatter(bl_bleu[:min_len], lora_bleu[:min_len], alpha=0.4, s=20, color="#8e44ad")
max_v = max(max(bl_bleu[:min_len]) if bl_bleu[:min_len] else 0.01, max(lora_bleu[:min_len]) if lora_bleu[:min_len] else 0.01, 0.01)
axes[1].plot([0, max_v], [0, max_v], "k--", lw=1.5, label="y=x (equal performance)")
axes[1].set_title("Per-Sample BLEU: Baseline vs LoRA"); axes[1].set_xlabel("Baseline BLEU")
axes[1].set_ylabel("LoRA BLEU"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Flan-T5-Small Science Q&A: BLEU Score Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/bleu_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"All extended visualizations saved to {CONFIG['output_dir']}/")

In [ ]:
results = {
    "student": "Avinash Megavatn (16829749)", "config": CONFIG,
    "baseline_metrics": baseline_metrics, "lora_metrics": lora_metrics,
    "training_time_seconds": train_time, "total_parameters": total_params,
    "trainable_parameters": trainable, "parameter_reduction_percent": round(reduction, 2),
    "epoch_metrics": epoch_metrics,
}
with open(f"{CONFIG['output_dir']}/results.json", "w") as f:
    json.dump(results, f, indent=2)
comparison.to_csv(f"{CONFIG['output_dir']}/comparison.csv", index=False)
model.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
tokenizer.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
print(f"All results saved to {CONFIG['output_dir']}/")